# 03 · Fault signatures and feature resolution
**Goal:** compare faults with healthy context, inspect component localisation and identify synthetic shortcuts.

In [ ]:
from pathlib import Path
import os
import sys

# Works when Jupyter starts from the repository root or from nbs.
HELPERS = Path.cwd() if (Path.cwd() / "fleetguard_eda.py").exists() else Path.cwd() / "nbs"
sys.path.insert(0, str(HELPERS))
import fleetguard_eda as eda
import panel as pn
import plotly.express as px

eda.setup()
ROOT = eda.project_root()
SOURCE = Path(os.environ.get("FLEETGUARD_EDA_SOURCE", ROOT / "data/generated/portfolio-v1"))
FEATURES = Path(os.environ.get("FLEETGUARD_EDA_FEATURES", ROOT / "data/features/portfolio-v1"))
SPLIT = "train"  # change to validation to check findings; test is blocked for exploration
ws = eda.Workspace(SOURCE, FEATURES)
pn.pane.Markdown(f"**Source:** `{SOURCE}`  ·  **Features:** `{FEATURES}`  ·  **Explore:** `{SPLIT}`\n\nMaps need internet. Linked controls need a running, trusted notebook kernel.")

In [ ]:
eda.banner("03 · What changes during a fault", "Evidence → neighbours → context → raw/minute resolution → limitations")

### 1 · Actual observed evidence
Assigned fault wagons can contain healthy periods. Counts here are reports, not independent fault incidents.

In [ ]:
observed = ws.load("wagon", SPLIT)
evidence = observed.groupby(["variant", "anomaly_type"]).size().reset_index(name="Received reports")
eda.pane(eda.figure(px.bar(evidence, x="variant", y="Received reports", color="anomaly_type"), "Observed labels versus assigned variants — no test exploration"))

### 2 · Inspect each scenario
Choose a fault, level and signal. Match brake demand/speed band before comparing distributions. Target/parent context means localisation context, not that every displayed signal is faulty.

In [ ]:
eda.anomaly_explorer(ws, SPLIT)

### 3 · Raw versus minute summaries
Choose a signature. Shading marks the configured ramp/interval; persistent faults can continue beyond it. Minute values are plotted at window start, not at sensor acquisition time.

In [ ]:
resolution_options = {
    "Wheel slide": ("wheel_slide", "axle", "wheel_speed_kph"),
    "Locked axle": ("locked_axle", "axle", "wheel_speed_kph"),
    "Bearing degradation": ("bearing_degradation", "wheel", "bearing_temp_c"),
    "Sensor drift": ("sensor_fault", "wheel", "bearing_temp_c"),
    "Suspected flat": ("suspected_wheel_flat", "axle", "vibration_rms_g"),
    "Brake leak": ("brake_pressure_leak", "wagon", "brake_pipe_pressure_bar"),
}
resolution = pn.widgets.Select(name="Signature", options=resolution_options)
pn.Column(resolution, pn.bind(lambda choice: eda.pane(eda.resolution_view(ws, *choice, split=SPLIT)), resolution))

### 4 · Availability and outages
A valid zero, null sensor reading and absent report are different. Minute extrema cannot recover a completely unobserved transient.

In [ ]:
eda.table(eda.quality_profile(ws, SPLIT))

In [ ]:
eda.pane(eda.outage_view(ws, SPLIT))

### 5 · Geographic evidence
Heat layer = anomalous-report density. Circle tooltip = anomaly reports / received reports within rounded coordinate cells. These describe synthetic exposure, not calibrated operational risk.

In [ ]:
fault_map_choice = pn.widgets.Select(name="Fault map", options=["All", *sorted(observed.variant.unique())])
def fault_map_view(variant):
    selected = observed if variant == "All" else observed[observed.variant == variant]
    return pn.pane.HTML(eda.folium_map(selected, faults=True)._repr_html_(), height=620, sizing_mode="stretch_width")
pn.Column(fault_map_choice, pn.bind(fault_map_view, fault_map_choice))

### 6 · Synthetic shortcut audit
Timing/route/severity come from design metadata. Use them to challenge the simulator, never as predictive features.

In [ ]:
eda.artefact_views(ws, SPLIT)

### 7 · Record modelling decisions
- Which signatures require raw 10-second inputs?
- Which sensors/components change, and which stay unchanged?
- Are null/zero/outage distinctions preserved?
- Are magnitude, timing or route shortcuts too easy?
- Does the healthy comparison share relevant operating conditions?

**Next milestone:** explicit input and evaluation-label contracts. Confirm findings on validation; reserve test signal exploration for final evaluation.